# Masterliste Updater – Development Notebook

End-to-end pipeline that mirrors the Streamlit app:

1. **Setup** – import from `core`, define disk helpers
2. **Load** – orders CSV, contacts CSV, existing masterlists, PDF fee schedule
3. **Process** – parse PDF, merge, build new-members table
4. **Update** – deduplicate-append new rows to each masterliste and save
5. **Validate** – compare column formats between original and appended rows

In [ ]:
"""Setup: add project root to sys.path so `core` is importable, then import helpers."""
import sys
import pathlib

# Make the project root importable regardless of which directory the kernel uses
ROOT = pathlib.Path("..").resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd

from core.config import COL, ENCODING_FALLBACKS
from core.io import load_csv_bytes, df_to_csv_bytes
from core.pdf import parse_pdf_fees
from core.transform import parse_membership, build_new_members, slice_outputs

DATA = ROOT / "data"

# ── Notebook-only helpers ──────────────────────────────────────────────────────

def load_csv_path(path: pathlib.Path, sep: str = ",") -> pd.DataFrame:
    """Load a CSV from disk using the project's encoding fallback chain."""
    return load_csv_bytes(path.read_bytes(), path.name, sep=sep)


def dedup_append(
    existing: pd.DataFrame, new_rows: pd.DataFrame, label: str = ""
) -> pd.DataFrame:
    """Append *new_rows* to *existing*, skipping rows whose dedup key already appears.

    Dedup key: 'Email' if present, else 'Membership Number #'.
    Rows with a null/empty key are never considered duplicates (always kept).
    Existing rows always win (keep='first').
    """
    for key in ("Email", "Membership Number #"):
        if key in existing.columns and key in new_rows.columns:
            break
    else:
        print(f"  ⚠  {label}: no dedup key found – appending without dedup")
        return pd.concat([existing, new_rows], ignore_index=True)

    combined = pd.concat([existing, new_rows], ignore_index=True)
    has_key  = combined[key].notna() & (combined[key].astype(str).str.strip() != "")
    deduped  = combined[has_key].drop_duplicates(subset=[key], keep="first")
    no_key   = combined[~has_key]
    result   = pd.concat([deduped, no_key]).sort_index().reset_index(drop=True)

    skipped = len(combined) - len(result)
    if skipped:
        print(f"  ℹ  {label}: {skipped} duplicate(s) skipped (matched on '{key}')")
    return result


print("Setup complete – core modules loaded from", ROOT)


=== PDF Page 1 ===
['ESPN & IPNA', 'IPNA', 'Membership', 'Note']
['140 €', '0 €', 'ESPN', '']
['192 €', '52 €', 'ESPN&IPNA online journal', '']
['248 €', '108 €', 'ESPN&IPNA online&print journal', '']
['100 €', '0 €', 'ESPN lower-middle income', '']


In [ ]:
# ── 1. Load source exports (comma-separated) ──────────────────────────────────
orders   = load_csv_path(DATA / "orders.csv")
contacts = load_csv_path(DATA / "contacts.csv")

# ── 2. Load existing masterlists (semicolon-separated) ────────────────────────
ipna_ml = load_csv_path(DATA / "masterliste_ipna.csv",       sep=";")
neue_ml = load_csv_path(DATA / "masterliste_new_member.csv", sep=";")
voll_ml = load_csv_path(DATA / "masterliste_all.csv",        sep=";")

print(f"Orders:    {len(orders)} rows   | columns: {orders.columns.tolist()}")
print(f"Contacts:  {len(contacts)} rows  | columns: {contacts.columns.tolist()}")
print(f"IPNA ML:   {len(ipna_ml)} rows")
print(f"Neue ML:   {len(neue_ml)} rows")
print(f"Voll ML:   {len(voll_ml)} rows")

# ── 3. Parse PDF fee schedule ──────────────────────────────────────────────────
fee_lookup, mem_col, espn_ipna_col, ipna_amt_col = parse_pdf_fees(
    (DATA / "beitrage_2026.pdf").read_bytes()
)
print(f"\nPDF columns: membership='{mem_col}', espn_ipna='{espn_ipna_col}', ipna='{ipna_amt_col}'")
print("Fee types:", list(fee_lookup.keys()))

# ── 4. Merge orders + contacts → new_members ──────────────────────────────────
email_l, email_r = COL["email"], COL["contact_email"]
for col, df, name in [(email_l, orders, "orders"), (email_r, contacts, "contacts")]:
    if col not in df.columns:
        raise KeyError(f"Email column '{col}' not found in {name}. Available: {df.columns.tolist()}")

merged = orders.merge(contacts, left_on=email_l, right_on=email_r, how="left")
merged["Membership"] = merged[COL["item"]].apply(parse_membership)

new_members = build_new_members(merged, fee_lookup, ipna_amt_col)
print(f"\nNew members to add: {len(new_members)}")
display(new_members)

PDF columns detected: membership='Membership', espn_ipna='ESPN & IPNA', ipna='IPNA'
Fee schedule loaded: ['ESPN', 'ESPN&IPNA online journal', 'ESPN&IPNA online&print journal', 'ESPN lower-middle income', 'ESPN&IPNA lower-middle income - online journal', 'ESPN&IPNA lower-middle income - online&print journal', 'ESPN low income', 'ESPN&IPNA low income - online journal', 'ESPN&IPNA low income - online&print journal', 'ESPN Membership 2026 Honorary Members', 'ESPN / ESPN lower-middle income / ESPN low income']

New members to add: 1


,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,IPNA amount,Membership,Gender,Note
0,11929,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €","52,00 €",ESPN&IPNA lower-middle income - online journal,NaN,


  ℹ  IPNA Masterliste: 1 duplicate(s) skipped (matched on 'Email')

IPNA Masterliste: 3 → 3 rows


,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,IPNA amount,Membership,Gender,Note
0,Associate Professor,IULIANA MAGDALENA,STARCEA,magdabirm@yahoo.com,'+40 726 704 612,10.07.1974,"62-64, V. Lupu street",Ia?i,700648,Romania,Ia?i,"Pediatric Nephrology Department, Grigore T. Po...",10.01.2024 17:47,"152,00 Û","52,00 Û",ESPN&IPNA lower-middle income - online journal,Female,NaN
1,Dr.,Bohuslav,Gibl‡k,giblakbohuslav@gmail.com,'+421 944 364 707,13.06.1989,Meden‡ 10,Bansk‡ Bystrica,97405,Slovakia,Banska Bystrica,Children hospital - Detsk‡ fakultn‡ nemocnica ...,15.05.2024 19:39,"152,00 Û","52,00 Û",ESPN&IPNA lower-middle income - online journal,Male,NaN
2,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",52 €,ESPN&IPNA lower-middle income - online journal,NaN,NaN


In [ ]:
# ── Update: IPNA Masterliste ───────────────────────────────────────────────────
ipna_new, _, _ = slice_outputs(new_members)   # columns already filtered to COLS_IPNA

updated_ipna = dedup_append(ipna_ml, ipna_new, label="IPNA Masterliste")
updated_ipna.to_csv(DATA / "masterliste_ipna.csv", sep=";", index=False, encoding="windows-1252")

print(f"IPNA Masterliste:     {len(ipna_ml)} → {len(updated_ipna)} rows")
display(updated_ipna.tail(len(ipna_new) + 2))

  ℹ  Masterliste neue Mitglieder: 1 duplicate(s) skipped (matched on 'Email')
Masterliste neue Mitglieder: 3 → 3 rows


,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,Membership,Gender,Note
0,11924,Assoc Prof,Serra,Sźrmeli Dšven,serrasurmel@yahoo.com,'+90 505 919 98 86,17.11.1982,?smet ?nšnź Bulvarő,Yeni?ehir,33120,Turkey,Mersin,Mersin University Faculty of Medicine,27.05.2024 06:08,"140,00 Ű",ESPN,Female,NaN
1,11925,Dr.,Manuela,Colucci,manuela.colucci@opbg.net,'+39 320 760 7919,04.12.1980,"Via Luigi Canina, 6",Roma,196,Italy,Lazio,Bambino Gesť Children Hospital (IRCCS),25.03.2024 16:05,"140,00 Ű",ESPN,Female,NaN
2,11929,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",ESPN&IPNA lower-middle income - online journal,NaN,NaN


In [ ]:
# ── Update: Masterliste neue Mitglieder ────────────────────────────────────────
_, neue_new, _ = slice_outputs(new_members)   # columns already filtered to COLS_NEUE

updated_neue = dedup_append(neue_ml, neue_new, label="Masterliste neue Mitglieder")
updated_neue.to_csv(DATA / "masterliste_new_member.csv", sep=";", index=False, encoding="windows-1250")

print(f"Masterliste neue Mitglieder:  {len(neue_ml)} → {len(updated_neue)} rows")
display(updated_neue.tail(len(neue_new) + 2))

  ℹ  Masterliste Vollständige Liste: 1 duplicate(s) skipped (matched on 'Email')
Masterliste Vollständige Liste: 4 → 4 rows


,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,Membership,Gender,Note
1,11928.0,Dr.,Rik,Westland,ri.westland@amsterdamumc.nl,'+31 6 50063988,08.11.1985,Meibergdreef 9,Amsterdam,1105 AZ,Netherlands,Noord-Holland,Amsterdam UMC,02.01.2024 18:52,"192,00 Û",ESPN&IPNA online journal,Male,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11929.0,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",ESPN&IPNA lower-middle income - online journal,NaN,NaN


In [ ]:
# ── Update: Masterliste vollständige Liste ────────────────────────────────────
_, _, voll_new = slice_outputs(new_members)   # columns already filtered to COLS_VOLL

updated_voll = dedup_append(voll_ml, voll_new, label="Masterliste vollständige Liste")
updated_voll.to_csv(DATA / "masterliste_all.csv", sep=";", index=False, encoding="windows-1252")

print(f"Masterliste vollständige Liste:  {len(voll_ml)} → {len(updated_voll)} rows")
display(updated_voll.tail(len(voll_new) + 2))


  Masterliste_full.csv
  Columns missing : —
  Extra columns   : —
  Column order OK : True

  Column                         orig dtype   new dtype    orig pattern           new pattern
  ---------------------------------------------------------------------------------------------------------
  Membership Number #            float64      float64      free text              free text
  Titel                          str          str          free text              (empty)  ← CHECK
  First Name                     str          str          free text              (empty)  ← CHECK
  Last Name                      str          str          free text              (empty)  ← CHECK
  Email                          str          str          free text              free text
  Phone                          str          str          free text              (empty)  ← CHECK
  Birthdate                      str          str          free text              (empty)  ← CHECK
  Address                

,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,Membership,Gender,Note
636,11929.0,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",ESPN&IPNA lower-middle income - online journal,NaN,



  Masterliste neue Mitglieder.csv
  Columns missing : —
  Extra columns   : —
  Column order OK : True

  Column                         orig dtype   new dtype    orig pattern           new pattern
  ---------------------------------------------------------------------------------------------------------
  Membership Number #            int64        int64        numeric string         numeric string
  Titel                          str          str          free text              (empty)  ← CHECK
  First Name                     str          str          free text              (empty)  ← CHECK
  Last Name                      str          str          free text              (empty)  ← CHECK
  Email                          str          str          free text              free text
  Phone                          str          str          free text              (empty)  ← CHECK
  Birthdate                      str          str          free text              (empty)  ← CHECK
  Address

,Membership Number #,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,Membership,Gender,Note
3,11929,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €",ESPN&IPNA lower-middle income - online journal,NaN,



  IPNA Masterliste.csv
  Columns missing : —
  Extra columns   : —
  Column order OK : True

  Column                         orig dtype   new dtype    orig pattern           new pattern
  ---------------------------------------------------------------------------------------------------------
  Titel                          str          str          free text              (empty)  ← CHECK
  First Name                     str          str          free text              (empty)  ← CHECK
  Last Name                      str          str          free text              (empty)  ← CHECK
  Email                          str          str          free text              free text
  Phone                          str          str          free text              (empty)  ← CHECK
  Birthdate                      str          str          free text              (empty)  ← CHECK
  Address                        str          str          free text              free text
  City                   

,Titel,First Name,Last Name,Email,Phone,Birthdate,Address,City,Zipcode,Country,State,Company,Member since,ESPN&IPNA amount,IPNA amount,Membership,Gender,Note
4,NaN,NaN,NaN,nprintza@gmail.com,NaN,NaN,Konstantinoupoleos 49,Thessaloniki,"""546 42""",GRC,GR-54,NaN,"Mar 1, 2026","152,00 €","52,00 €",ESPN&IPNA lower-middle income - online journal,NaN,


---
## Format validation

Checks that the appended rows match the column schema and value patterns of the original masterlists.

In [ ]:
def compare_format(original: pd.DataFrame, appended: pd.DataFrame, label: str) -> None:
    """Compare appended rows against the original schema: columns, dtypes, value patterns."""
    orig_cols = original.columns.tolist()
    new_rows  = appended.iloc[len(original):]

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")

    missing_in_new = [c for c in orig_cols if c not in appended.columns]
    extra_in_new   = [c for c in appended.columns if c not in orig_cols]
    order_ok       = appended.columns.tolist() == orig_cols
    print(f"  Columns missing : {missing_in_new or '—'}")
    print(f"  Extra columns   : {extra_in_new   or '—'}")
    print(f"  Column order OK : {order_ok}")

    def sample_pattern(series: pd.Series) -> str:
        s = series.dropna().astype(str).str.strip()
        s = s[s != ""]
        if s.empty:
            return "(empty)"
        if s.str.match(r"^\d{1,3}(,\d{3})*(\.\d+)? €$").mean() > 0.5:
            return "euro-amount"
        if s.str.match(r"^\d{4}-\d{2}-\d{2}").mean() > 0.5:
            return "date (YYYY-MM-DD)"
        if s.str.match(r"^\d+$").mean() > 0.7:
            return "numeric string"
        return "free text"

    print(f"\n  {'Column':<30} {'orig dtype':<12} {'new dtype':<12} {'orig pattern':<22} new pattern")
    print(f"  {'-'*100}")
    for col in orig_cols:
        if col not in appended.columns:
            continue
        orig_pat = sample_pattern(original[col])
        new_pat  = sample_pattern(new_rows[col]) if not new_rows.empty else "(no new rows)"
        dtype_ok = "✓" if original[col].dtype == appended[col].dtype else "≠"
        pat_ok   = "✓" if orig_pat == new_pat else "≠"
        flag     = "" if dtype_ok == "✓" and pat_ok == "✓" else "  ← CHECK"
        print(f"  {col:<30} {str(original[col].dtype):<12} {str(appended[col].dtype):<12} {orig_pat:<22} {new_pat}{flag}")

    if not new_rows.empty:
        print(f"\n  Sample of appended rows ({min(3, len(new_rows))} of {len(new_rows)}):")
        display(new_rows.head(3))
    else:
        print("\n  ⚠  No new rows were appended.")


compare_format(ipna_ml, updated_ipna, "masterliste_ipna.csv")
compare_format(neue_ml, updated_neue, "masterliste_new_member.csv")
compare_format(voll_ml, updated_voll, "masterliste_all.csv")